# Step 2: Fine-tuning GPT-2 for Product Ad Copy Generation

## Overview
This notebook fine-tunes **`GPT-2`** on the **`llm-wizard/Product-Descriptions-and-Ads`** dataset.

The fine-tuned model learns to generate compelling advertising copy given a product name and description. In the final application, the product description is supplied by the BLIP caption model from Step 1.

### Pipeline
```
Product Image → [BLIP] → Caption → [Fine-tuned GPT-2] → Ad Copy
```

### Dataset
- **Source**: `llm-wizard/Product-Descriptions-and-Ads`
- **Size**: 90 train / 10 test
- **Format**: `product` + `description` → `ad`

### Model Options
| Model | Params | Notes |
|-------|--------|-------|
| `GPT-2` | 117M | Causal LM, fast to fine-tune |
| `t5-base` | 220M | Encoder-decoder, better for conditional generation |

This notebook implements **both** options; select `USE_T5 = True/False` below.

In [ ]:
!pip install -q transformers datasets accelerate huggingface_hub evaluate rouge_score

In [ ]:
import os
import torch
import numpy as np
from datasets import load_dataset
from torch.utils.data import Dataset
from transformers import (
    GPT2Tokenizer, GPT2LMHeadModel,
    T5Tokenizer, T5ForConditionalGeneration,
    AutoTokenizer, AutoModelForCausalLM,
    Trainer, TrainingArguments,
    Seq2SeqTrainer, Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq, DataCollatorForLanguageModeling,
)
import evaluate

# ── Configuration ─────────────────────────────────────────────────────────
USE_T5   = False            # True → T5-base  |  False → GPT-2
DEVICE   = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device : {DEVICE}")
print(f"Model  : {'T5-base' if USE_T5 else 'GPT-2'}")

In [ ]:
from huggingface_hub import login

HF_TOKEN   = "hf_YOUR_TOKEN_HERE"    # <── HuggingFace write token
HF_USERNAME = "JescYip"

if USE_T5:
    BASE_MODEL = "t5-base"
    MODEL_REPO = f"{HF_USERNAME}/t5-ad-finetuned"
else:
    BASE_MODEL = "gpt2"
    MODEL_REPO = f"{HF_USERNAME}/gpt2-ad-finetuned"

login(token=HF_TOKEN)
print(f"Base model : {BASE_MODEL}")
print(f"Push target: {MODEL_REPO}")

## 1. Load & Explore Dataset

In [ ]:
raw = load_dataset("llm-wizard/Product-Descriptions-and-Ads")
train_raw = raw["train"]
test_raw  = raw["test"]
print(f"Train: {len(train_raw)} | Test: {len(test_raw)}")
print(f"Columns: {train_raw.column_names}")

In [ ]:
print("Sample entries:\n" + "="*60)
for i in range(3):
    r = train_raw[i]
    print(f"[{i+1}] Product    : {r['product'].strip()}")
    print(f"     Description: {r['description'].strip()}")
    print(f"     Ad         : {r['ad'].strip()}")
    print()

## 2. Text Formatting

### GPT-2 format (causal LM — trained on the full sequence)
```
Product: {product}\nDescription: {description}\nAd: {ad}<|endoftext|>
```
At inference: provide `Product: X\nDescription: Y\nAd:` → GPT-2 completes the ad.

### T5 format (encoder-decoder)
- **Input** : `generate ad: Product: {product} Description: {description}`
- **Target**: `{ad}`

In [ ]:
def format_gpt2(row):
    """Full training text for GPT-2 (causal LM)."""
    return (
        f"Product: {row['product'].strip()}\n"
        f"Description: {row['description'].strip()}\n"
        f"Ad: {row['ad'].strip()}<|endoftext|>"
    )

def inference_prompt_gpt2(product, description=""):
    """Prompt for GPT-2 inference (stop before the ad)."""
    base = f"Product: {product.strip()}\n"
    if description:
        base += f"Description: {description.strip()}\n"
    return base + "Ad:"

def format_t5_input(row):
    return (
        f"generate ad: Product: {row['product'].strip()} "
        f"Description: {row['description'].strip()}"
    )

# Preview
r = train_raw[0]
print("=== GPT-2 Training Text ===")
print(format_gpt2(r))
print()
print("=== T5 Input ===")
print(format_t5_input(r))
print("T5 Target:", r['ad'].strip())

## 3. Tokeniser & Model Loading

In [ ]:
if USE_T5:
    tokenizer = T5Tokenizer.from_pretrained(BASE_MODEL)
    model     = T5ForConditionalGeneration.from_pretrained(BASE_MODEL).to(DEVICE)
else:
    tokenizer = GPT2Tokenizer.from_pretrained(BASE_MODEL)
    # GPT-2 has no pad token — set eos as pad
    tokenizer.pad_token = tokenizer.eos_token
    model = GPT2LMHeadModel.from_pretrained(BASE_MODEL).to(DEVICE)
    model.config.pad_token_id = tokenizer.eos_token_id

total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model      : {BASE_MODEL}")
print(f"Parameters : {total:,} total | {trainable:,} trainable")

## 4. PyTorch Dataset

In [ ]:
# ── GPT-2 Dataset ─────────────────────────────────────────────────────────
class GPT2AdDataset(Dataset):
    """
    For GPT-2 causal LM: encodes the full formatted text.
    Labels = input_ids (next-token prediction on the entire sequence).
    """
    def __init__(self, hf_data, tokenizer, max_length=256):
        self.encodings = []
        for row in hf_data:
            text = format_gpt2(row)
            enc  = tokenizer(
                text,
                truncation=True,
                max_length=max_length,
                padding="max_length",
                return_tensors="pt",
            )
            input_ids      = enc["input_ids"].squeeze()
            attention_mask = enc["attention_mask"].squeeze()
            # Labels same as input_ids; padding masked with -100
            labels = input_ids.clone()
            labels[attention_mask == 0] = -100
            self.encodings.append({
                "input_ids":      input_ids,
                "attention_mask": attention_mask,
                "labels":         labels,
            })

    def __len__(self):  return len(self.encodings)
    def __getitem__(self, i): return self.encodings[i]


# ── T5 Dataset ────────────────────────────────────────────────────────────
class T5AdDataset(Dataset):
    """
    For T5 encoder-decoder: encodes input and target separately.
    """
    def __init__(self, hf_data, tokenizer, max_input=256, max_target=128):
        self.encodings = []
        for row in hf_data:
            src = format_t5_input(row)
            tgt = row["ad"].strip()
            model_inputs = tokenizer(
                src, max_length=max_input,
                truncation=True, padding="max_length", return_tensors="pt"
            )
            with tokenizer.as_target_tokenizer():
                labels = tokenizer(
                    tgt, max_length=max_target,
                    truncation=True, padding="max_length", return_tensors="pt"
                )
            label_ids = labels["input_ids"].squeeze().clone()
            label_ids[label_ids == tokenizer.pad_token_id] = -100
            self.encodings.append({
                "input_ids":      model_inputs["input_ids"].squeeze(),
                "attention_mask": model_inputs["attention_mask"].squeeze(),
                "labels":         label_ids,
            })

    def __len__(self):  return len(self.encodings)
    def __getitem__(self, i): return self.encodings[i]


# ── Build datasets ────────────────────────────────────────────────────────
if USE_T5:
    train_dataset = T5AdDataset(train_raw, tokenizer)
    val_dataset   = T5AdDataset(test_raw,  tokenizer)
else:
    train_dataset = GPT2AdDataset(train_raw, tokenizer)
    val_dataset   = GPT2AdDataset(test_raw,  tokenizer)

print(f"Train: {len(train_dataset)} | Val: {len(val_dataset)}")
sample = train_dataset[0]
print("Keys:", list(sample.keys()))
print("input_ids shape:", sample["input_ids"].shape)

# Decode first sample to verify
print("\nDecoded sample:")
ids = sample["input_ids"]
print(tokenizer.decode(ids[ids != -100][:80], skip_special_tokens=False))

## 5. Fine-Tuning

In [ ]:
# ── Evaluation: ROUGE ─────────────────────────────────────────────────────
rouge = evaluate.load("rouge")

def compute_metrics(eval_pred):
    preds, labels = eval_pred
    if isinstance(preds, tuple):
        preds = preds[0]
    pad_id = tokenizer.pad_token_id
    preds  = np.where(preds  != -100, preds,  pad_id)
    labels = np.where(labels != -100, labels, pad_id)
    decoded_preds  = tokenizer.batch_decode(preds,  skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    decoded_labels = [[l] for l in decoded_labels]
    result = rouge.compute(
        predictions=decoded_preds,
        references=decoded_labels,
        use_stemmer=True
    )
    return {k: round(v, 4) for k, v in result.items()}

In [ ]:
OUTPUT_DIR = f"./{BASE_MODEL.replace('/', '-')}-ad-finetuned"

# ── Training arguments (shared) ───────────────────────────────────────────
common_kwargs = dict(
    output_dir=OUTPUT_DIR,
    num_train_epochs=20,          # Small dataset → more epochs
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    warmup_steps=10,
    learning_rate=3e-4,
    weight_decay=0.01,
    logging_dir=f"{OUTPUT_DIR}/logs",
    logging_steps=10,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="rougeL",
    fp16=(DEVICE == "cuda"),
    report_to="none",
    push_to_hub=True,
    hub_model_id=MODEL_REPO,
    hub_token=HF_TOKEN,
)

if USE_T5:
    args = Seq2SeqTrainingArguments(
        predict_with_generate=True,
        generation_max_length=128,
        **common_kwargs
    )
    trainer = Seq2SeqTrainer(
        model=model,
        args=args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        tokenizer=tokenizer,
        compute_metrics=compute_metrics,
    )
else:
    args = TrainingArguments(**common_kwargs)
    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        tokenizer=tokenizer,
    )

print("Starting fine-tuning...")
trainer.train()

## 6. Evaluation & Inference

In [ ]:
model.eval()

print("Generating ads for test set:")
print("=" * 60)

for row in test_raw:
    product     = row["product"].strip()
    description = row["description"].strip()
    gt_ad       = row["ad"].strip()

    if USE_T5:
        src    = format_t5_input(row)
        inputs = tokenizer(src, return_tensors="pt", truncation=True, max_length=256).to(DEVICE)
        with torch.no_grad():
            out = model.generate(
                **inputs,
                max_new_tokens=100,
                num_beams=4,
                early_stopping=True,
            )
        generated = tokenizer.decode(out[0], skip_special_tokens=True)
    else:
        prompt  = inference_prompt_gpt2(product, description)
        inputs  = tokenizer(prompt, return_tensors="pt").to(DEVICE)
        with torch.no_grad():
            out = model.generate(
                **inputs,
                max_new_tokens=80,
                do_sample=True,
                temperature=0.8,
                top_p=0.9,
                pad_token_id=tokenizer.eos_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )
        full_text = tokenizer.decode(out[0], skip_special_tokens=True)
        # Extract only the ad part
        generated = full_text.split("Ad:")[-1].strip().split("\n")[0]

    print(f"Product   : {product}")
    print(f"Ground truth: {gt_ad[:80]}")
    print(f"Generated : {generated[:80]}")
    print()

## 7. Save & Push to HuggingFace Hub

In [ ]:
# ── Save locally ──────────────────────────────────────────────────────────
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Model saved to {OUTPUT_DIR}")

# ── Push to Hub ───────────────────────────────────────────────────────────
model.push_to_hub(MODEL_REPO, token=HF_TOKEN)
tokenizer.push_to_hub(MODEL_REPO, token=HF_TOKEN)
print(f"\n✅ Model pushed to: https://huggingface.co/{MODEL_REPO}")

In [ ]:
# ── Verify from Hub ───────────────────────────────────────────────────────
from transformers import pipeline as hf_pipeline

if USE_T5:
    gen = hf_pipeline("text2text-generation", model=MODEL_REPO, token=HF_TOKEN)
    test_input = format_t5_input(test_raw[0])
    result = gen(test_input, max_new_tokens=80)
    print("Hub result:", result[0]["generated_text"])
else:
    gen = hf_pipeline(
        "text-generation", model=MODEL_REPO, token=HF_TOKEN,
        pad_token_id=50256
    )
    test_prompt = inference_prompt_gpt2(
        test_raw[0]["product"], test_raw[0]["description"]
    )
    result = gen(test_prompt, max_new_tokens=80, do_sample=True,
                 temperature=0.8, top_p=0.9)
    full   = result[0]["generated_text"]
    ad     = full.split("Ad:")[-1].strip().split("\n")[0]
    print(f"Hub model ad: {ad}")

print(f"\n✅ GPT-2/T5 fine-tuning complete!")
print(f"Model: https://huggingface.co/{MODEL_REPO}")

## 8. App Integration

Update `app_refined.py` to use the fine-tuned models:

```python
# Step 1: BLIP generates caption from product image
from transformers import BlipProcessor, BlipForConditionalGeneration

blip_processor = BlipProcessor.from_pretrained("JescYip/blip-fashion-finetuned")
blip_model = BlipForConditionalGeneration.from_pretrained("JescYip/blip-fashion-finetuned")

inputs = blip_processor(images=uploaded_image, return_tensors="pt")
caption_ids = blip_model.generate(**inputs, max_new_tokens=50)
caption = blip_processor.decode(caption_ids[0], skip_special_tokens=True)

# Step 2: GPT-2 generates ad from caption
from transformers import GPT2Tokenizer, GPT2LMHeadModel

gpt2_tokenizer = GPT2Tokenizer.from_pretrained("JescYip/gpt2-ad-finetuned")
gpt2_model = GPT2LMHeadModel.from_pretrained("JescYip/gpt2-ad-finetuned")

prompt = f"Product: {caption}\nAd:"
inputs = gpt2_tokenizer(prompt, return_tensors="pt")
outputs = gpt2_model.generate(**inputs, max_new_tokens=80,
                               do_sample=True, temperature=0.8,
                               pad_token_id=50256)
ad = gpt2_tokenizer.decode(outputs[0], skip_special_tokens=True)
ad = ad.split("Ad:")[-1].strip()
```

## Notes

| Parameter | GPT-2 | T5-base |
|-----------|-------|---------|
| Architecture | Decoder-only | Encoder-Decoder |
| Training loss | Causal LM | Seq2Seq |
| Dataset size | 90 samples | 90 samples |
| Epochs | 20 | 20 |
| Learning rate | 3e-4 | 3e-4 |
| Batch size | 8 | 8 |